# Polynomial Regression으로 Calories_Burned 예측
각 fold에서 Polynomial과 Scaling을 train fold로만 학습한 뒤, 

원단위 Ridge와 sqrt-Ridge 두 모델의 OOF 예측을 생성하고, 

최적 가중치로 결합하여 최종 예측

최종 RMSE : 0.29

## 1. 라이브러리 임포트, 랜덤 시드 , 상수 고정

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

RANDOM_STATE = 42
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

## 2. 데이터 로드

In [3]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
submission = pd.read_csv("sample_submission.csv")

## 3. Feature Engineering Function - 수정금지

In [5]:
def create_features_safe(df):
    df = df.copy()

    df['Height_Total_Inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']
    df['Temp_diff'] = df['Body_Temperature(F)'] - 98.6

    df['Duration_bin'] = pd.cut(
        df['Exercise_Duration'],
        bins=[-np.inf, 10, 20, 30, np.inf],
        labels=[0, 1, 2, 3]
    ).astype(int)

    df['Duration_x_BPM'] = df['Exercise_Duration'] * df['BPM']
    df['Duration_x_TempDiff'] = df['Exercise_Duration'] * df['Temp_diff']
    df['BPM_x_TempDiff'] = df['BPM'] * df['Temp_diff']

    # 최종 TOP5에 쓰는 핵심 변수명 고정
    df["Intensity"] = df["Duration_x_BPM"]
    df["Effort"] = df["Weight(lb)"] * df["Intensity"]

    df['Duration_sq'] = df['Exercise_Duration'] ** 2
    df['Temp_diff_sq'] = df['Temp_diff'] ** 2
    df['Dur_BPM_TempDiff'] = df['Exercise_Duration'] * df['BPM'] * df['Temp_diff']

    df['BPM_per_Duration'] = df['BPM'] / (df['Exercise_Duration'] + 1)
    df['TempDiff_per_Duration'] = df['Temp_diff'] / (df['Exercise_Duration'] + 1)

    h2 = (df['Height_Total_Inches'] ** 2).replace(0, np.nan)
    df['BMI'] = 703 * df['Weight(lb)'] / h2
    df['BMI'] = df['BMI'].fillna(df['BMI'].median())

    df['Weight_x_Duration'] = df['Weight(lb)'] * df['Exercise_Duration']

    df['Log_BPM'] = np.log1p(df['BPM'])
    df['Log_Duration'] = np.log1p(df['Exercise_Duration'])
    df['Log_Weight_BPM_Dur'] = np.log1p(df['Weight(lb)'] * df['BPM'] * df['Exercise_Duration'])

    return df

### Preprocess (train-fit encoder) + Reduced feature 
최종 학습에 사용할 feature 집합 정리

* generated_cols는 파생변수 목록

* reduced는 base(원핫 포함) + TOP5

In [ ]:
# 0) raw split (타겟 분리 원본피처만 남기기) - 모델 입력은 X와 y로 분리되어야 하고 ID는 예측에 필요 없기 때문
train_x_raw = train.drop(['ID', 'Calories_Burned'], axis=1).copy()
train_y = train['Calories_Burned'].astype(float).copy()
test_x_raw  = test.drop(['ID'], axis=1).copy()

# 1) feature engineering
before_cols = list(train_x_raw.columns)
train_x_feat = create_features_safe(train_x_raw)
test_x_feat  = create_features_safe(test_x_raw)

generated_cols = [c for c in train_x_feat.columns if c not in before_cols]  # 파생변수만

# 2) one-hot (train fit -> test transform)
cat_cols = [c for c in ['Gender', 'Weight_Status'] if c in train_x_feat.columns]
if len(cat_cols) > 0:
    enc = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
    enc.fit(train_x_feat[cat_cols]) # train에서만 fit 누수방지

    tr_cat = enc.transform(train_x_feat[cat_cols])
    te_cat = enc.transform(test_x_feat[cat_cols])
    cat_names = enc.get_feature_names_out(cat_cols)

    train_x_feat = train_x_feat.drop(columns=cat_cols)
    test_x_feat  = test_x_feat.drop(columns=cat_cols)

    train_x_feat[cat_names] = tr_cat
    test_x_feat[cat_names]  = te_cat

train_x = train_x_feat.copy()
test_x  = test_x_feat.copy()

# 3) reduced = base + TOP5 (Pruning)
TOP5_FIXED = ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity'] # 중요도기반 5개만
base_feats = [c for c in train_x.columns if c not in generated_cols]  # 원핫 포함 base 유지 (원본 + 원핫)

keep_cols = base_feats + TOP5_FIXED
train_red = train_x[keep_cols].copy()
test_red  = test_x[keep_cols].copy()

print("Final train_x/test_x:", train_x.shape, test_x.shape)
print("Reduced train/test:", train_red.shape, test_red.shape)
print("TOP5 present:", [c for c in TOP5_FIXED if c in train_red.columns])

Final train_x/test_x: (7500, 28) (7500, 28)
Reduced train/test: (7500, 15) (7500, 15)
TOP5 present: ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity']


## Train + Fine-tuned alpha + weight (최종)
KFold OOF + Test fold-averaging

	1.	OOF RMSE 계산 - 진짜 일반화 성능을 보기 위해(누수 방지)
	2.	Test 예측 생성 - fold마다 학습한 모델의 test 예측을 평균내서 더 안정화
	3.	두 베이스 모델을 섞어서 최종 예측 - identity 타겟 모델 + sqrt 타겟 모델을 convex weight로 결합


In [ ]:
DEGREE = 3  # PolynomialFeatures로 3차까지 확장 (곡선 관계 표현)
ALPHA_ID = 0.05 # identity(원단위) Ridge 규제 강도 : 작을수록 규제가 약해져 표현력이 증가  pruning 후 약하게 하는 게 최적이었음
ALPHA_SQ = 0.009    # sqrt 타겟 Ridge 규제 강도 보조역할로 넣은거라 값 작음
W_ID = 0.9851676708561896   # 최종 결합 가중치(거의 identity 위주)

# 타겟 준비
y = train_y.values.astype(float)    #원단위 타겟
y_sqrt = np.sqrt(np.clip(y, 0, None))   #sqrt 변환 타겟 
            # clip 음수 방지(혹시몰라 안전장치용)        #sqrt는 큰 값(고칼로리) 쪽을 상대적으로 눌러서 학습하게 만들어, 일부 구간에서 안정화를 기대하는 변환

# OOF/Test 예측 배열 생성
# OOF는 fold별 validation index에만 채워지고, test는 fold마다 예측해서 평균냄
id_test = np.zeros(len(test_red))   #  identity 모델의 test 예측 누적(나중에 fold 평균)
sq_test = np.zeros(len(test_red))   # sqrt 모델의 test 예측 누적
id_oof = np.zeros(len(train_red))   #  identity 모델의 OOF 예측 저장 공간 (train 길이)
sq_oof = np.zeros(len(train_red))   # sqrt 모델의 OOF 예측 저장 공간

# KFold 루프 - 이 구조 덕분에 OOF가 누수 없이 생성됨
for tr_idx, va_idx in kf.split(train_red):
    #fold 데이터 분리
    X_tr_raw = train_red.iloc[tr_idx]   # train_red: 이미 (base + TOP5)로 pruning된 최종 입력, fold마다 train/val로 나눔
    X_va_raw = train_red.iloc[va_idx]

    # PolynomialFeatures (fold 내부 fit)
    poly = PolynomialFeatures(degree=DEGREE, include_bias=False)
    X_tr = poly.fit_transform(X_tr_raw) # fit_transform은 오직 X_tr_raw에서만! 누수방지용
    X_va = poly.transform(X_va_raw)
    X_te = poly.transform(test_red)

    # StandardScaler (fold 내부 fit) 스케일러도 train fold에서만 fit
    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    X_te_s = sc.transform(X_te)

    # Base 모델 1: identity Ridge
    m_id = Ridge(alpha=ALPHA_ID, random_state=RANDOM_STATE)
    m_id.fit(X_tr_s, y[tr_idx]) # 학습 타겟: 원단위 y
    id_oof[va_idx] = m_id.predict(X_va_s)   # OOF 예측: validation index 위치에 저장
    id_test += m_id.predict(X_te_s) / N_SPLITS  # test 예측: fold마다 평균 → / N_SPLITS로 누적

    # Base 모델 2: sqrt Ridge
    m_sq = Ridge(alpha=ALPHA_SQ, random_state=RANDOM_STATE)
    m_sq.fit(X_tr_s, y_sqrt[tr_idx])    # 	학습 타겟: sqrt(y)

    # 예측값은 sqrt 스케일이니까 다시 원단위로 복원해야 함
    pred_va_sq = m_sq.predict(X_va_s)
    pred_te_sq = m_sq.predict(X_te_s)
                        # clip은 음수 예측 방지(제곱하면 이상해질 수 있음) 복원은 **2
    sq_oof[va_idx] = np.clip(pred_va_sq, 0, None) ** 2
    sq_test += (np.clip(pred_te_sq, 0, None) ** 2) / N_SPLITS

# 칼로리는 음수가 있을 수 없으니 안전하게 0 이상으로 고정
id_oof = np.clip(id_oof, 0, None)
sq_oof = np.clip(sq_oof, 0, None)
id_test = np.clip(id_test, 0, None)
sq_test = np.clip(sq_test, 0, None)

# 최종 결합 (가중 합)\
# W_ID ≈ 0.985라서 거의 identity가 메인이지만, sqrt가 아주 미세하게 보정 (특정 구간에서 RMSE를 조금 깎아줌)
final_oof = np.clip(W_ID * id_oof + (1 - W_ID) * sq_oof, 0, None)
final_test = np.clip(W_ID * id_test + (1 - W_ID) * sq_test, 0, None)

print("OOF RMSE:", rmse(y, final_oof))
print("Test stats mean/min/max:", float(final_test.mean()), float(final_test.min()), float(final_test.max()))

OOF RMSE: 0.29936955597925347
Test stats mean/min/max: 89.70246112500861 0.572990129195138 313.34637579921815


In [ ]:
submission["Calories_Burned"] = final_test
submission.to_csv("submit_final_0299.csv", index=False)
print(" 저장 : submit_final_0299.csv")

## 최종 RMSE : 0.29936955597925347

## 제출 후 리더보드 점수 : 0.29964